# 02 · Repeated validation and CYP inhibition error analysis

**Research question:** Are the apparent weaknesses of uncertainty-based CYP inhibition prediction reproducible across validation splits, and do the largest errors occur near the extremes of experimental pIC50?

This standalone, **Google Colab CPU-compatible** notebook extends [Notebook 01](01_cyp_uncertainty_study.ipynb). It runs multiple random and Bemis–Murcko scaffold-grouped splits using the publicly available OpenADMET CYP direct-inhibition training dataset. No blind-test labels are used.

**Predefined hypotheses**
1. **H1 — Activity-range compression.** The model predicts a narrower distribution than the measured one and displays signed errors at extreme activities.
2. **H2 — Uncertainty ranking.** Bootstrap-ensemble disagreement and nearest-neighbor chemical distance are weak or inconsistent predictors of error, especially under scaffold shift.
3. **H3 — Experimental precision.** Dose-response confidence-interval width is associated with prediction error, but is **not** a prospective input at inference time.

**Run:** Runtime → Run all. Dependencies are installed automatically. The first execution uses **five seeds, three ensemble members, and 80 trees/member**; a smaller optional quick-run setting is provided below. Plots and CSV tables are saved under `results/notebook_02/` and packaged into a ZIP for easy download.

> **Interpretation:** Repeated splits share compounds, so summaries across folds show *sensitivity to split choice*, not independent confidence intervals or significance tests. Scaffold splits approximate, but do not reproduce, the challenge's hidden chemical-series evaluation.

## 1 · Install dependencies and configure the experiment

Change `SEEDS`, `N_ENSEMBLE` and `N_TREES` to balance CPU runtime against robustness. For a quick smoke test, set `SEEDS=[2026, 2027]`, `N_ENSEMBLE=2`, `N_TREES=30`. Return to the predefined configuration before reporting findings.

In [ ]:
import os
if os.environ.get("CYP_SKIP_INSTALL", "0") != "1":
    get_ipython().run_line_magic("pip", '-q install "rdkit>=2023.9" "scikit-learn>=1.3" "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "scipy>=1.10"')

In [ ]:
from pathlib import Path
import json, platform, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
import sklearn
from IPython.display import display

# Experimental protocol: fix before comparing model families.
SEEDS = [2026, 2027, 2028, 2029, 2030]
N_ENSEMBLE = 3
N_TREES = 80
TEST_FRACTION = 0.20
RADIUS, N_BITS = 2, 2048
CYP_ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
COVERAGES = np.linspace(0.10, 1.0, 10)

# For a local automated smoke test; not used in ordinary Colab runs.
if os.environ.get('CYP_FAST_TEST') == '1':
    SEEDS, N_ENSEMBLE, N_TREES = [2026, 2027], 2, 12

OUT = Path('results/notebook_02')
OUT.mkdir(parents=True, exist_ok=True)
print('Configuration:', dict(seeds=SEEDS, members=N_ENSEMBLE, trees=N_TREES,
                             test_fraction=TEST_FRACTION))
print('Runtime:', platform.python_version(), 'RDKit:', rdBase.rdkitVersion,
      'sklearn:', sklearn.__version__)

## 2 · Download the data and validate the column mappings

This uses the same OpenADMET public CSV and target definitions as Notebook 01. Set `CYP_DATA_PATH` to a local CSV path if you have already downloaded the file. The notebook **stops with an explicit error** if expected pIC50 or confidence-interval columns cannot be found; it will not silently use a TDI endpoint.

In [ ]:
DATA_URL = ('https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/'
            'resolve/main/cyp-challenge-TRAIN_inhibition.csv')
local_path = os.environ.get('CYP_DATA_PATH', '').strip()
source = local_path if local_path else DATA_URL
raw = pd.read_csv(source)

SMILES_COL = 'SMILES'
TARGETS = {iso: f'{iso}_pIC50_direct_inhibition' for iso in CYP_ISOFORMS}
CIS = {iso: (f'{TARGETS[iso]}_conf_low', f'{TARGETS[iso]}_conf_high')
       for iso in CYP_ISOFORMS}
required = [SMILES_COL] + list(TARGETS.values())
missing = [col for col in required if col not in raw.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}. Actual: {list(raw.columns)}')
for iso, (lo, hi) in CIS.items():
    if (lo in raw.columns) != (hi in raw.columns):
        raise ValueError(f'Incomplete CI column pair for {iso}: {lo}, {hi}')

print('Input shape:', raw.shape)
print('CYP targets:')
for iso in CYP_ISOFORMS:
    print('  ', iso, TARGETS[iso], 'observed:',
          pd.to_numeric(raw[TARGETS[iso]], errors='coerce').notna().sum(),
          'CI:', all(c in raw.columns for c in CIS[iso]))
display(raw[[SMILES_COL] + list(TARGETS.values())].head())

## 3 · Canonicalize chemistry and build one fingerprint matrix

All Morgan fingerprints and molecular identities are derived **only from SMILES**, so their construction does not depend on labels. To prevent exact-structure leakage, random splits group identical canonical SMILES together. Scaffold validation groups Bemis–Murcko scaffold keys; acyclic compounds use their canonical SMILES instead of being combined into one artificial supergroup.

In [ ]:
start = time.time()
mols0 = [Chem.MolFromSmiles(str(s)) if pd.notna(s) else None for s in raw[SMILES_COL]]
keep = np.array([m is not None for m in mols0], dtype=bool)
work = raw.loc[keep].reset_index(drop=True).copy()
mols = [m for m, ok in zip(mols0, keep) if ok]
canon = np.array([Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
                  for m in mols], dtype=object)

def murcko_key(m):
    scaffold = MurckoScaffold.GetScaffoldForMol(m)
    return (Chem.MolToSmiles(scaffold, canonical=True, isomericSmiles=True)
            if scaffold.GetNumAtoms() else 'ACYCLIC::' + Chem.MolToSmiles(m, canonical=True, isomericSmiles=True))
scaffolds = np.array([murcko_key(m) for m in mols], dtype=object)

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)
fps = [fp_gen.GetFingerprint(m) for m in mols]
X = np.empty((len(fps), N_BITS), dtype=np.uint8)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, X[i])
print(f'Valid molecules {len(work):,}/{len(raw):,} | exact-structure groups {len(set(canon)):,} | '
      f'Murcko/acyclic groups {len(set(scaffolds)):,} | fingerprints {X.shape} | '
      f'featurization {time.time()-start:.1f}s')

Y = {iso: pd.to_numeric(work[TARGETS[iso]], errors='coerce').to_numpy(dtype=float)
     for iso in CYP_ISOFORMS}
CI_VALUES = {}
for iso in CYP_ISOFORMS:
    lo_col, hi_col = CIS[iso]
    if lo_col in work.columns and hi_col in work.columns:
        lo = pd.to_numeric(work[lo_col], errors='coerce').to_numpy(dtype=float)
        hi = pd.to_numeric(work[hi_col], errors='coerce').to_numpy(dtype=float)
        CI_VALUES[iso] = (lo, hi)
    else:
        CI_VALUES[iso] = (np.full(len(work), np.nan), np.full(len(work), np.nan))

## 4 · Predefine splits, uncertainty measures, and outcome diagnostics

- **Random:** 20% of canonical-structure groups held out.
- **Scaffold:** 20% of Murcko/acyclic groups held out (the number of held-out *molecules* may differ).
- **Ensemble uncertainty:** standard deviation of predictions from bootstrap random-forest ensemble members.
- **Similarity uncertainty:** `1 − max(Tanimoto(query, training molecule))`.
- **Combined uncertainty:** unweighted mean of the **within-fold percentile ranks** of the two signals. This ranking is fixed in advance; it is not fit to validation outcomes.
- **Interval excess:** distance outside the *measured* 95% confidence interval; zero if the prediction lies within it. This is a **local diagnostic, not the official MA-ST-RAE**.
- **Activity extremes:** bottom 10%, middle 50% (25th–75th percentiles), and top 10% of experimental pIC50 **within each validation fold**. These strata use outcome labels for retrospective diagnosis, never to choose which molecules to predict.

In [ ]:
def split_indices(y, grouping, seed):
    observed = np.flatnonzero(np.isfinite(y))
    if len(observed) < 30:
        raise ValueError('Too few measured training compounds for a meaningful split.')
    group_subset = grouping[observed]
    if len(np.unique(group_subset)) < 3:
        raise ValueError('Too few unique groups for grouped validation.')
    cv = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_local, valid_local = next(cv.split(observed, groups=group_subset))
    train_id, valid_id = observed[train_local], observed[valid_local]
    assert len(set(grouping[train_id]) & set(grouping[valid_id])) == 0
    assert len(set(canon[train_id]) & set(canon[valid_id])) == 0, 'Canonical SMILES leakage!'
    return train_id, valid_id


def fit_bootstrap_forest(train_id, valid_id, y, seed):
    rng = np.random.default_rng(seed)
    y_train = y[train_id]
    preds = []
    for member in range(N_ENSEMBLE):
        sample = rng.integers(0, len(train_id), size=len(train_id))
        model = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=2,
                                      max_features=0.7, random_state=seed + member,
                                      n_jobs=-1)
        model.fit(X[train_id[sample]], y_train[sample])
        preds.append(model.predict(X[valid_id]))
    preds = np.vstack(preds)
    return preds.mean(axis=0), preds.std(axis=0, ddof=1) if N_ENSEMBLE > 1 else np.zeros(len(valid_id))


def max_training_similarity(train_id, valid_id):
    train_fps = [fps[i] for i in train_id]
    return np.asarray([max(DataStructs.BulkTanimotoSimilarity(fps[i], train_fps))
                       for i in valid_id], dtype=float)


def interval_excess(yhat, lo, hi):
    val = np.maximum(lo - yhat, 0.0) + np.maximum(yhat - hi, 0.0)
    valid = np.isfinite(lo) & np.isfinite(hi) & (lo <= hi)
    return np.where(valid, val, np.nan)


def activity_groups(y):
    q10, q25, q75, q90 = np.quantile(y, [0.10, 0.25, 0.75, 0.90])
    return np.select([y <= q10, (y >= q25) & (y <= q75), y >= q90],
                     ['lowest_10pct', 'middle_50pct', 'highest_10pct'], default='other')


def risk_coverage_curve(y, yh, scores):
    order = np.argsort(np.asarray(scores), kind='stable')
    return [(float(c), int(max(1, np.ceil(c*len(order)))),
             float(mean_absolute_error(y[order[:max(1, int(np.ceil(c*len(order))))]],
                                       yh[order[:max(1, int(np.ceil(c*len(order))))]])))
            for c in COVERAGES]


def safe_spearman(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5 or np.std(x[mask]) == 0 or np.std(y[mask]) == 0:
        return np.nan
    return float(spearmanr(x[mask], y[mask]).statistic)

## 5 · Run all repeated validations

This is the computationally expensive cell. Predictions, outcomes, interval widths, and uncertainty signals are saved for every held-out compound. No thresholds, model parameters, or signal weights are tuned against validation outcomes. Change the configuration **once** if you need to shorten the run; report the actual values from metadata.

In [ ]:
rows, all_preds, all_risks, all_corrs, all_bins = [], [], [], [], []
started = time.time()
for iso_idx, iso in enumerate(CYP_ISOFORMS):
    y = Y[iso]
    lo_all, hi_all = CI_VALUES[iso]
    print(f'\n[{iso}] observed labels: {np.isfinite(y).sum():,}', flush=True)
    for split_kind in ('random', 'scaffold'):
        grouping = canon if split_kind == 'random' else scaffolds
        for seed in SEEDS:
            train_id, valid_id = split_indices(y, grouping, seed)
            # Separate bootstrap random streams across endpoints/splits/seeds.
            model_seed = int(seed + 100000*iso_idx + (50000 if split_kind == 'scaffold' else 0))
            yval = y[valid_id]
            yh, u_ensemble = fit_bootstrap_forest(train_id, valid_id, y, model_seed)
            sim = max_training_similarity(train_id, valid_id)
            u_similarity = 1 - sim
            ranks_ens = pd.Series(u_ensemble).rank(method='average', pct=True).to_numpy()
            ranks_sim = pd.Series(u_similarity).rank(method='average', pct=True).to_numpy()
            u_combined = 0.5 * (ranks_ens + ranks_sim)
            residual = yh - yval
            abs_error = np.abs(residual)
            lo, hi = lo_all[valid_id], hi_all[valid_id]
            valid_ci = np.isfinite(lo) & np.isfinite(hi) & (lo <= hi)
            ci_width = np.where(valid_ci, hi-lo, np.nan)
            ci_excess = interval_excess(yh, lo, hi)
            bins = activity_groups(yval)
            group_overlap = len(set(grouping[train_id]) & set(grouping[valid_id]))
            assert group_overlap == 0
            result = {
                'isoform': iso, 'split': split_kind, 'seed': seed,
                'n_train': len(train_id), 'n_valid': len(valid_id),
                'heldout_fraction': len(valid_id)/(len(train_id)+len(valid_id)),
                'mae': float(mean_absolute_error(yval, yh)),
                'r2': float(r2_score(yval, yh)),
                'mean_measured_pic50': float(np.mean(yval)),
                'sd_measured_pic50': float(np.std(yval, ddof=1)),
                'sd_predicted_pic50': float(np.std(yh, ddof=1)),
                'sd_ratio': float(np.std(yh, ddof=1)/np.std(yval, ddof=1)) if np.std(yval, ddof=1)>0 else np.nan,
                'slope_pred_vs_measured': float(np.polyfit(yval, yh, deg=1)[0]) if np.std(yval)>0 else np.nan,
                'mean_interval_excess': float(np.nanmean(ci_excess)) if valid_ci.any() else np.nan,
                'mean_ci_width': float(np.nanmean(ci_width)) if valid_ci.any() else np.nan,
                'n_valid_ci': int(valid_ci.sum()),
                'fraction_within_ci': float(np.mean(ci_excess[valid_ci]==0)) if valid_ci.any() else np.nan,
            }
            rows.append(result)
            pred = pd.DataFrame({
                'isoform': iso, 'split': split_kind, 'seed': seed,
                'row_index': valid_id, 'measured_pic50': yval, 'predicted_pic50': yh,
                'signed_residual': residual, 'abs_error': abs_error,
                'activity_group': bins, 'ensemble_std': u_ensemble,
                'max_tanimoto': sim, 'similarity_uncertainty': u_similarity,
                'combined_uncertainty': u_combined,
                'ci_low': lo, 'ci_high': hi, 'ci_width': ci_width,
                'interval_excess': ci_excess,
            })
            all_preds.append(pred)
            for signal in ('ensemble_std', 'similarity_uncertainty', 'combined_uncertainty'):
                score = pred[signal].to_numpy(dtype=float)
                all_corrs.append({'isoform': iso, 'split': split_kind, 'seed': seed,
                                  'uncertainty_signal': signal,
                                  'spearman_rho': safe_spearman(score, abs_error),
                                  'n_valid': len(valid_id)})
                for cov, nret, mae in risk_coverage_curve(yval, yh, score):
                    all_risks.append({'isoform': iso, 'split': split_kind, 'seed': seed,
                                      'uncertainty_signal': signal, 'coverage': cov,
                                      'n_retained': nret, 'mae_retained': mae,
                                      'mae_full': result['mae'], 'delta_mae_vs_full': mae-result['mae']})
            for grp in ('lowest_10pct', 'middle_50pct', 'highest_10pct'):
                sub = pred.loc[pred['activity_group'].eq(grp)]
                if len(sub):
                    all_bins.append({'isoform': iso, 'split': split_kind, 'seed': seed,
                                     'activity_group': grp, 'n': len(sub),
                                     'mae': float(sub['abs_error'].mean()),
                                     'mean_signed_residual': float(sub['signed_residual'].mean()),
                                     'mean_ci_width': float(sub['ci_width'].mean()) if sub['ci_width'].notna().any() else np.nan,
                                     'mean_interval_excess': float(sub['interval_excess'].mean()) if sub['interval_excess'].notna().any() else np.nan})
        print(f'  {split_kind}: {len(SEEDS)} splits complete; elapsed {(time.time()-started)/60:.1f} min', flush=True)

metrics = pd.DataFrame(rows)
predictions = pd.concat(all_preds, ignore_index=True)
risk = pd.DataFrame(all_risks)
correlations = pd.DataFrame(all_corrs)
activity = pd.DataFrame(all_bins)
assert len(metrics) == len(CYP_ISOFORMS) * 2 * len(SEEDS)
print('Completed in', round((time.time()-started)/60, 2), 'minutes')
display(metrics.groupby(['isoform', 'split'])[['mae','r2','sd_ratio','heldout_fraction']].agg(['mean','std']).round(3))

## 6 · Result A — How variable is MAE across splits?

The plot shows **mean ± standard deviation across seeds**, not confidence intervals. Held-out sample counts differ because scaffold splits are selected by scaffold group, not by identical numbers of molecules. Compare activity distributions before interpreting a difference in difficulty.

In [ ]:
summary = (metrics.groupby(['isoform','split'], as_index=False)
           .agg(n_splits=('seed','nunique'), mean_mae=('mae','mean'), sd_mae=('mae','std'),
                mean_r2=('r2','mean'), mean_sd_ratio=('sd_ratio','mean'),
                mean_validation_fraction=('heldout_fraction','mean'),
                sd_validation_fraction=('heldout_fraction','std')))
display(summary.round(3))
fig, ax = plt.subplots(figsize=(9,4.8))
x = np.arange(len(CYP_ISOFORMS))
for k, name in enumerate(('random','scaffold')):
    sub = summary[summary.split.eq(name)].set_index('isoform').reindex(CYP_ISOFORMS)
    xs = x + (-0.12 if name=='random' else 0.12)
    ax.errorbar(xs, sub.mean_mae, yerr=sub.sd_mae, capsize=4, marker='o', linestyle='none',
                label=name)
ax.set_xticks(x, CYP_ISOFORMS)
ax.set_ylabel('MAE ± SD across seeds (pIC50 units)')
ax.set_title('Repeated random vs scaffold-grouped validation')
ax.legend()
fig.tight_layout()
fig.savefig(OUT/'01_repeated_validation_mae.png', dpi=170)
plt.show()

## 7 · Result B — Regression to the mean and activity-dependent errors

`predicted pIC50 − measured pIC50` is **positive when inhibition potency is overpredicted** and negative when it is underpredicted. The scatterplots show one predefined seed for readability; the summary bars aggregate across all seeds. A low prediction-to-measurement SD ratio is evidence of **range compression**, but conditioning on measured outcomes can also create regression-to-the-mean patterns due to measurement noise; do not interpret the plots as proof of a causal mechanism.

In [ ]:
view = predictions[(predictions.split=='scaffold') & (predictions.seed==SEEDS[0])]
fig, axs = plt.subplots(2,2,figsize=(11,8), sharey=True)
for ax, iso in zip(axs.flat,CYP_ISOFORMS):
    v = view.loc[view.isoform.eq(iso)]
    ax.scatter(v.measured_pic50, v.signed_residual, alpha=.45, s=16)
    ax.axhline(0,color='black',linestyle='--',linewidth=1)
    ax.set_title(iso)
    ax.set_xlabel('Measured pIC50')
    ax.set_ylabel('Predicted − measured pIC50')
fig.suptitle(f'Scaffold validation residuals (seed {SEEDS[0]})')
fig.tight_layout()
fig.savefig(OUT/'02_residuals_by_activity.png',dpi=170)
plt.show()

activity_summary = (activity.groupby(['isoform','split','activity_group'],as_index=False)
                    .agg(mean_mae=('mae','mean'), sd_mae=('mae','std'),
                         mean_residual=('mean_signed_residual','mean'),
                         mean_ci_width=('mean_ci_width','mean'),
                         mean_excess=('mean_interval_excess','mean')))
display(activity_summary[activity_summary.split=='scaffold'].round(3))

fig, axs = plt.subplots(2,2,figsize=(11,8), sharey=True)
for ax, iso in zip(axs.flat,CYP_ISOFORMS):
    g = activity_summary[(activity_summary.isoform==iso)&(activity_summary.split=='scaffold')]
    order = ['lowest_10pct','middle_50pct','highest_10pct']
    g = g.set_index('activity_group').reindex(order)
    ax.bar(np.arange(3),g.mean_mae,yerr=g.sd_mae,capsize=4)
    ax.set_xticks(np.arange(3),['Lowest 10%','Middle 50%','Highest 10%'])
    ax.set_title(iso)
    ax.set_ylabel('Mean MAE ± SD across seeds')
fig.suptitle('Activity-stratified errors: scaffold split')
fig.tight_layout()
fig.savefig(OUT/'03_activity_stratified_mae.png',dpi=170)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
for iso in CYP_ISOFORMS:
    g = metrics[(metrics.isoform==iso)&(metrics.split=='scaffold')]
    ax.scatter(np.repeat(iso,len(g)),g.sd_ratio,alpha=.65,label=iso)
ax.axhline(1,color='black',linestyle='--',label='Equal predicted/measured SD')
ax.set_ylabel('SD(predicted pIC50) / SD(measured pIC50)')
ax.set_title('Distributional compression across scaffold splits')
fig.tight_layout()
fig.savefig(OUT/'04_prediction_variance_ratio.png',dpi=170)
plt.show()

## 8 · Result C — Do uncertainty signals rank errors reliably?

Higher Spearman rho is better: an uncertainty score should increase when absolute error increases. Every point below is one fold; average and SD summarize reproducibility across five seeds. Negative rho means the apparent uncertainty signal ranks errors **in the wrong direction** on that fold. These correlations describe rank ordering; they do not establish calibrated probabilities.

In [ ]:
corr_summary = (correlations.groupby(['isoform','split','uncertainty_signal'],as_index=False)
                .agg(mean_rho=('spearman_rho','mean'),sd_rho=('spearman_rho','std'),
                     n_seeds=('seed','nunique')))
display(corr_summary.round(3))
fig,axs=plt.subplots(2,2,figsize=(12,7),sharey=True)
signals=['ensemble_std','similarity_uncertainty','combined_uncertainty']
for ax,iso in zip(axs.flat,CYP_ISOFORMS):
    sub=corr_summary[(corr_summary.isoform==iso)&(corr_summary.split=='scaffold')].set_index('uncertainty_signal').reindex(signals)
    pos=np.arange(len(signals))
    ax.errorbar(pos,sub.mean_rho,yerr=sub.sd_rho,fmt='o',capsize=4)
    ax.axhline(0,color='gray',linestyle='--',linewidth=1)
    ax.set_xticks(pos,['Ensemble','Similarity','Combined'])
    ax.set_title(iso)
    ax.set_ylabel('Spearman rho (mean ± SD)')
fig.suptitle('Uncertainty–error rank associations under scaffold shift')
fig.tight_layout()
fig.savefig(OUT/'05_uncertainty_correlations.png',dpi=170)
plt.show()

## 9 · Result D — Risk–coverage curves, repeated over seeds

Retain the lowest-uncertainty molecules first. A useful score produces **lower MAE at partial coverage** than at 100% coverage. To avoid artificially making combined uncertainty look better, its weights were fixed at 50:50 before examining validation outcomes. Shaded bands are across-seed SD, not confidence intervals.

In [ ]:
risk_summary=(risk.groupby(['isoform','split','uncertainty_signal','coverage'],as_index=False)
              .agg(mean_mae=('mae_retained','mean'), sd_mae=('mae_retained','std'),
                   mean_delta=('delta_mae_vs_full','mean')))
fig,axs=plt.subplots(2,2,figsize=(12,8),sharey=False)
for ax,iso in zip(axs.flat,CYP_ISOFORMS):
    sub=risk_summary[(risk_summary.isoform==iso)&(risk_summary.split=='scaffold')]
    for sig in signals:
        g=sub[sub.uncertainty_signal.eq(sig)].sort_values('coverage')
        xx=g.coverage.to_numpy(); yy=g.mean_mae.to_numpy(); sd=g.sd_mae.fillna(0).to_numpy()
        ax.plot(xx,yy,marker='.',label=sig.replace('_uncertainty','').replace('_std',''))
        ax.fill_between(xx,yy-sd,yy+sd,alpha=.12)
    baseline=metrics[(metrics.isoform==iso)&(metrics.split=='scaffold')]['mae'].mean()
    ax.axhline(baseline,color='gray',linestyle='--',linewidth=1)
    ax.set_title(iso)
    ax.set_xlabel('Fraction of predictions retained')
    ax.set_ylabel('MAE (lower is better)')
axs[0,0].legend(fontsize=8)
fig.suptitle('Scaffold-split risk–coverage across repeated seeds')
fig.tight_layout()
fig.savefig(OUT/'06_repeated_risk_coverage.png',dpi=170)
plt.show()
# Fixed 50% coverage is an interpretable summary alongside curves.
risk50 = (risk[np.isclose(risk.coverage,0.50)]
          .groupby(['isoform','split','uncertainty_signal'],as_index=False)
          .agg(mean_mae_50=('mae_retained','mean'),
               sd_mae_50=('mae_retained','std'),
               mean_improvement_50=('delta_mae_vs_full',lambda z: -float(np.mean(z)))))
display(risk50.round(3))

## 10 · Result E — Experimental uncertainty and confidence intervals

The supplied `*_conf_low`/`*_conf_high` values describe uncertainty in measured pIC50, **not a measured property of unseen test molecules**. They are analyzed here exclusively after prediction as retrospective labels/diagnostics.

This plot separates raw absolute error from *interval excess* (distance beyond the reported interval). If a large raw error remains outside a narrow experimental interval, it is harder to explain using reported measurement uncertainty alone.

In [ ]:
ci_rows=[]
for (iso,split,seed),g in predictions.groupby(['isoform','split','seed']):
    ok=g['ci_width'].notna() & g['interval_excess'].notna()
    s=g.loc[ok]
    ci_rows.append({'isoform':iso,'split':split,'seed':seed,
                    'n_ci':len(s),
                    'rho_ci_width_vs_error':safe_spearman(s.ci_width,s.abs_error),
                    'mean_ci_width':float(s.ci_width.mean()) if len(s) else np.nan,
                    'mean_interval_excess':float(s.interval_excess.mean()) if len(s) else np.nan,
                    'fraction_pred_within_ci':float((s.interval_excess==0).mean()) if len(s) else np.nan})
ci_summary=pd.DataFrame(ci_rows)
display(ci_summary.groupby(['isoform','split'])[['rho_ci_width_vs_error','mean_ci_width',
       'mean_interval_excess','fraction_pred_within_ci']].agg(['mean','std']).round(3))

fig,axs=plt.subplots(2,2,figsize=(11,8),sharey=True)
for ax,iso in zip(axs.flat,CYP_ISOFORMS):
    g=activity_summary[(activity_summary.isoform==iso)&(activity_summary.split=='scaffold')]
    g=g.set_index('activity_group').reindex(['lowest_10pct','middle_50pct','highest_10pct'])
    idx=np.arange(3)
    ax.bar(idx-0.2,g.mean_mae,width=0.4,label='Raw absolute error')
    ax.bar(idx+0.2,g.mean_excess,width=0.4,label='Distance outside experimental CI')
    ax.set_xticks(idx,['Low 10%','Middle 50%','High 10%'])
    ax.set_title(iso); ax.set_ylabel('Mean pIC50 error')
axs[0,0].legend(fontsize=8)
fig.suptitle('Experimental interval versus apparent model error: scaffold splits')
fig.tight_layout()
fig.savefig(OUT/'07_interval_aware_error_by_activity.png',dpi=170)
plt.show()

## 11 · Save outputs and download a results ZIP

The results are separate from Notebook 01 so your original analysis is preserved. **Do not infer statistical independence from repeated validation rows; validation compounds can reappear across seeds.** The saved metadata records reproducibility settings and dependencies.

In [ ]:
outputs = {
    'repeated_split_metrics.csv':metrics,
    'split_summary.csv':summary,
    'repeated_predictions.csv':predictions,
    'activity_stratified_errors.csv':activity,
    'activity_stratified_summary.csv':activity_summary,
    'uncertainty_correlations.csv':correlations,
    'uncertainty_correlation_summary.csv':corr_summary,
    'risk_coverage_repeated.csv':risk,
    'risk_coverage_summary.csv':risk_summary,
    'risk_at_50pct_coverage.csv':risk50,
    'experimental_ci_diagnostics.csv':ci_summary,
}
for file,table in outputs.items():
    table.to_csv(OUT/file,index=False)
metadata={
    'notebook':'02_repeated_validation_error_analysis.ipynb',
    'dataset_url':DATA_URL if not local_path else 'local_override',
    'dataset_rows':int(len(raw)), 'valid_structures':int(len(work)),
    'target_columns':TARGETS,'ci_columns':CIS,
    'seeds':SEEDS,'ensemble_members':N_ENSEMBLE,'trees_per_member':N_TREES,
    'test_fraction_by_group':TEST_FRACTION,'radius':RADIUS,'n_bits':N_BITS,
    'random_grouping':'canonical SMILES',
    'scaffold_grouping':'Murcko scaffold; acyclic compounds canonical SMILES',
    'combined_uncertainty':'mean of within-fold percentile ranks of ensemble_std and (1 - maximum Tanimoto)',
    'python':platform.python_version(),'rdkit':rdBase.rdkitVersion,
    'sklearn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__,
    'notes':'Repeated splits overlap. Local CI interval excess is not the official challenge score.'
}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
archive=Path('cyp_notebook_02_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(OUT.iterdir()):
        if file.is_file():
            zf.write(file,arcname=f'notebook_02/{file.name}')
print('Saved',len(outputs),'data tables plus',len(list(OUT.glob('*.png'))),'figures')
print('ZIP:',archive.resolve(),'size:',round(archive.stat().st_size/1024/1024,2),'MiB')
if 'google.colab' in __import__('sys').modules:
    from google.colab import files
    # Uncomment the next line if you want Colab to immediately download the ZIP.
    # files.download(str(archive))

## 12 · Checklist for interpreting the new results

Before writing a scientific claim:

- [ ] Are **MAE differences between random/scaffold splits larger than seed-to-seed variation**? Report both the mean and SD, and variation in held-out sample counts.
- [ ] Are the **predicted pIC50 standard deviations systematically smaller** than experimental standard deviations across most folds?
- [ ] Do **signed residuals** reverse between the low- and high-potency tails across all CYPs and seeds?
- [ ] Are the **uncertainty–error Spearman correlations** consistently positive, near zero, or negative under scaffold shift?
- [ ] Does filtering to 50% coverage *reliably lower* MAE (mean improvement and its fold-to-fold variability)?
- [ ] Are large raw errors still large **after accounting for experimental confidence intervals**?
- [ ] Do conclusions survive changes to scaffold-grouping rules, model choices, and potential activity censoring?

**Avoid overclaiming:** activity-dependent residuals and shrinkage are expected to some degree for regression trained on noisy labels; the analysis identifies patterns but does not determine a causal explanation. Experimental CI width must not be treated as a known feature for blinded compounds. If promising patterns persist, the next project is chemical similarity and activity-cliff analysis, followed by an alternative regression model.

**Cite original resources:** [OpenADMET CYP dataset](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test), [official activity-prediction tutorial](https://github.com/OpenADMET/CYP-Challenge-Tutorial/blob/main/notebooks/activity_prediction.ipynb).